In [ ]:
# 1. Install required libraries

!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 147.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 49.1 MB/s eta 0:00:00


In [ ]:
import torch
import os
import gc
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig  # <-- IMPORTING THE NEW SFTConfig
from google.colab import drive

In [ ]:
# 1. THE FIX: Flush the corrupted model out of GPU memory
print("[INFO] Clearing old state from memory...")
try:
    del model
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

[INFO] Clearing old state from memory...


In [ ]:
# 2. Mount Google Drive
drive.mount('/content/drive')
drive_path = "/content/drive/My Drive/AI_Research_Assistant"
data_path = f"{drive_path}/mistral_training_data.jsonl"
output_dir = f"{drive_path}/mistral-7b-research-assistant"

Mounted at /content/drive


In [ ]:
# 3. Hugging Face Login (Required for Mistral)
from huggingface_hub import notebook_login
print("[INFO] Please log in to Hugging Face to access Mistral (ensure you have accepted their terms on the HF website):")
notebook_login()

[INFO] Please log in to Hugging Face to access Mistral (ensure you have accepted their terms on the HF website):


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# 4. Load Dataset
print("[INFO] Loading dataset from Google Drive...")
dataset = load_dataset("json", data_files=data_path, split="train")

[INFO] Loading dataset from Google Drive...


In [ ]:

# 5. Configure 4-bit Quantization for A100
print("[INFO] Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


[INFO] Configuring 4-bit quantization...


In [ ]:
# 4. Load Base Model and Tokenizer
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
print(f"[INFO] Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

[INFO] Loading mistralai/Mistral-7B-Instruct-v0.2...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
# 5. Configure LoRA Adapter
print("[INFO] Applying LoRA Adapter...")
peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

[INFO] Applying LoRA Adapter...


In [ ]:
# 6. Set Training Arguments (Using the NEW SFTConfig)
print("[INFO] Setting up SFTConfig...")
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    logging_steps=10,
    num_train_epochs=2,
    max_steps=-1,
    fp16=False,
    bf16=True,
    dataset_text_field="text",
    max_length=1024
)

[INFO] Setting up SFTConfig...


In [ ]:
# 7. Execute Trainer
print("[INFO] Initiating Training Loop...")
trainer = SFTTrainer(
    model=model,                # We pass the raw prepared model
    train_dataset=dataset,
    peft_config=peft_config,    # The Trainer applies the PEFT config for you natively!
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

[INFO] Initiating Training Loop...


Adding EOS to train dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/257 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.425491
20,0.794680
30,0.602168


TrainOutput(global_step=34, training_loss=0.9013490817126106, metrics={'train_runtime': 170.376, 'train_samples_per_second': 3.017, 'train_steps_per_second': 0.2, 'total_flos': 1.5290079220604928e+16, 'train_loss': 0.9013490817126106})

In [ ]:
# 10. Save Adapter to Drive
print(f"[INFO] Saving trained adapter to {output_dir}...")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("[SUCCESS] Fine-tuning complete. Adapter is safe in Google Drive!")

[INFO] Saving trained adapter to /content/drive/My Drive/AI_Research_Assistant/mistral-7b-research-assistant...
[SUCCESS] Fine-tuning complete. Adapter is safe in Google Drive!
